# 2.6 · 假设检验 / Hypothesis Testing

> **课程定位**
> Part 2 的**中心枢纽**：把前 5 课（SE、抽样分布、CI）组装成"做决策的机器"。覆盖检验框架、t 检验全家、卡方、ANOVA、非参检验，以及 **p 值的正确与错误打开方式**。
> The hub of Part 2: assembling SE + sampling distributions + CIs into a decision machine.

> 💡 **面试相关**
> - "解释 p 值" ★★★★★（与 CI 并列的必考题，错误答案满天飞）
> - "第一类/第二类错误" ★★★★★
> - "什么时候用非参检验" ★★★★
> - "配对 vs 独立 t 检验" ★★★★
> - "方差不等怎么办（Welch）" ★★★

---

## 目录
1. [检验框架：法庭审判模型 ⭐](#1)
2. [⭐ p 值：定义、误读、模拟验证](#2)
3. [单样本 t 检验（手写 + scipy 对照）](#3)
4. [两样本 t：为什么默认 Welch ⭐](#4)
5. [配对 t：方差削减的免费午餐](#5)
6. [比例检验（A/B 的原生形态）](#6)
7. [卡方检验：拟合优度 + 独立性](#7)
8. [ANOVA：三组以上](#8)
9. [非参检验：Mann-Whitney / Wilcoxon / KS](#9)
10. [实战：Tips 数据集检验套餐](#10)
11. [小结 + 检验选择决策树](#11)


<a id="1"></a>
## 1. 检验框架：法庭审判模型 ⭐ / The Courtroom Model

| 法庭 / Court | 统计 / Statistics |
|---|---|
| 无罪推定 | $H_0$（原假设：无效应/无差异）|
| 控方主张 | $H_1$（备择假设）|
| 证据 | 数据 → 检验统计量 |
| "排除合理怀疑" | $p < \alpha$ |
| 误判好人 | **Type I**（假阳性，概率 $\alpha$）|
| 放走罪犯 | **Type II**（假阴性，概率 $\beta$）|
| 定罪力 | **Power** $= 1 - \beta$（2.8 节主角）|

**流程**：设 $H_0$ → 选统计量 → 算"若 $H_0$ 真，数据这么极端的概率"（p 值）→ $p < \alpha$ 则拒绝 $H_0$。

⚠ **不拒绝 ≠ 接受 $H_0$**——"证据不足"不是"证明无罪"。样本太小时什么都测不出来（power 不足），不能反过来声称"证明了没差异"。
Failing to reject is "insufficient evidence", never "proof of no effect" — underpowered tests detect nothing.


<a id="2"></a>
## 2. ⭐ p 值：定义、误读、模拟验证 / The p-value

$$p = \Pr\big(\text{统计量} \ge \text{观测值} \;\big|\; H_0 \text{ 为真}\big)$$

**正确读法**："假设没效应，看到这么极端（或更极端）数据的概率。"

| ❌ 误读 | 为什么错 |
|---|---|
| "$H_0$ 为真的概率是 p" | p 是 $\Pr(\text{data}\mid H_0)$，不是 $\Pr(H_0 \mid \text{data})$——方向反了（贝叶斯才能给后者，2.10）|
| "p=0.04 → 96% 概率有效应" | 同上 |
| "p 越小效应越大" | p 混合了效应量和样本量；n 超大时微小效应也 p<0.001 |
| "p=0.06 失败, p=0.04 成功" | 0.05 是人为惯例；两者证据强度几乎相同 |

**关键性质（面试加分）**：$H_0$ 为真时 **p 值服从 Uniform(0,1)**——所以才会有 5% 的假阳性（p<0.05 的概率恰是 5%）。验证：
Under a true null, the p-value is Uniform(0,1) — that's exactly where the 5% false-positive rate comes from.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# H0 为真时 p ~ Uniform(0,1) / p is uniform under the null
pvals_null = [st.ttest_ind(rng.normal(0, 1, 50), rng.normal(0, 1, 50)).pvalue
              for _ in range(10_000)]
# H1 为真时 p 堆积在小值 / under the alternative, p piles up near 0
pvals_alt = [st.ttest_ind(rng.normal(0, 1, 50), rng.normal(0.5, 1, 50)).pvalue
             for _ in range(10_000)]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(pvals_null, bins=40, density=True, alpha=0.8)
axes[0].axhline(1, color="r", ls="--"); axes[0].set_title("H0 true: p ~ Uniform(0,1)")
axes[1].hist(pvals_alt, bins=40, density=True, alpha=0.8, color="C1")
axes[1].set_title("H1 true (effect=0.5σ): p piles near 0")
plt.tight_layout(); plt.show()

print(f"H0 真: P(p<0.05) = {np.mean(np.array(pvals_null) < 0.05):.3f}  (= α ✓)")
print(f"H1 真: P(p<0.05) = {np.mean(np.array(pvals_alt) < 0.05):.3f}  (= power, 2.8 节主角)")


<a id="3"></a>
## 3. 单样本 t 检验 / One-sample t-test

**问题**："样本均值和某个基准 $\mu_0$ 有差异吗？"

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}} \;\sim\; t_{n-1} \;\;(\text{under } H_0)$$

正是 2.3 的 SE + 2.5 的 t 分布拼起来——**检验和 CI 是同一枚硬币**：$p < 0.05 \iff \mu_0$ 落在 95% CI 外。


In [ ]:
# 手写 vs scipy / By hand vs scipy
x = rng.normal(102.5, 10, 40)          # 真均值 102.5, 检验 H0: μ=100
mu0 = 100.0

t_stat = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(len(x)))
p_manual = 2 * st.t.sf(abs(t_stat), df=len(x)-1)        # 双侧: 2×单尾 / two-sided

res = st.ttest_1samp(x, mu0)
print(f"手写:  t = {t_stat:.4f},  p = {p_manual:.4f}")
print(f"scipy: t = {res.statistic:.4f},  p = {res.pvalue:.4f}   ← 完全一致")

# 单侧检验: H1: μ > 100 / one-sided
print(f"单侧 p = {st.t.sf(t_stat, len(x)-1):.4f}  (双侧的一半; 方向必须在看数据前定!)")


> ⚠ **单侧检验的纪律**：方向必须**在看数据之前**由业务逻辑决定。看完数据再选方向 = p 值偷偷减半 = 作弊（2.7 节的 p-hacking 家族成员）。
> One-sided tests demand the direction be fixed before seeing data — choosing after is silent p-halving.


<a id="4"></a>
## 4. 两样本 t：为什么默认 Welch ⭐ / Two-sample: Default to Welch

| | Student（合并方差）| **Welch** ⭐ |
|---|---|---|
| 假设 | $\sigma_A = \sigma_B$ | 不要求 |
| SE | $s_p\sqrt{\tfrac{1}{n_A}+\tfrac{1}{n_B}}$ | $\sqrt{\tfrac{s_A^2}{n_A}+\tfrac{s_B^2}{n_B}}$ |
| 方差相等时 | 最优 | **几乎一样好** |
| 方差不等 + n 不等时 | **Type I 失控**（可到 15%+）| 仍然正确 |

**结论：永远用 Welch**（`equal_var=False`）。"先做方差齐性检验再决定"是过时流程——预检验本身引入问题。R 的 `t.test` 默认就是 Welch；scipy 默认还是 Student（历史包袱），**必须手动指定**。
Always Welch. R defaults to it; scipy doesn't — set `equal_var=False` yourself.


In [ ]:
# 演示 Student 的 Type I 失控 / Student's Type I inflation
# 方差不等 (1 vs 3) + 样本不等 (100 vs 25), H0 为真 (均值都是 0)
n_sim = 20_000
fp_student = fp_welch = 0
for _ in range(n_sim):
    a = rng.normal(0, 1.0, 100)
    b = rng.normal(0, 3.0, 25)         # 小组反而方差大 = 最坏情形 / worst case
    fp_student += st.ttest_ind(a, b, equal_var=True).pvalue < 0.05
    fp_welch   += st.ttest_ind(a, b, equal_var=False).pvalue < 0.05

print(f"名义 α = 5%, H0 为真:")
print(f"  Student t 假阳性率 = {fp_student/n_sim:.1%}   ← 失控!")
print(f"  Welch   t 假阳性率 = {fp_welch/n_sim:.1%}   ← 正常 ✓")


**Student 的假阳性率翻了 3 倍**（小组方差大时合并方差严重低估 SE）。一行 `equal_var=False` 的事。


<a id="5"></a>
## 5. 配对 t：方差削减的免费午餐 / Paired t-test

**场景**：同一对象测两次（减肥前后、同一查询新旧算法）。**对差值做单样本 t**：
$$t = \frac{\bar{d}}{s_d/\sqrt{n}}, \qquad d_i = x_i^{(\text{after})} - x_i^{(\text{before})}$$

**威力来源**：个体间差异（有人本来就重）在相减时**自我抵消**——
$$\mathrm{Var}(d) = \sigma_A^2 + \sigma_B^2 - 2\,\mathrm{Cov}(A, B)$$
相关越强，方差砍得越多。**能配对就配对**——这是 A/B 测试方差削减技术（CUPED, Part 19）的雏形。
The covariance term eats the between-subject noise. Pair whenever you can — this is the seed of CUPED.


In [ ]:
# 同一批人减肥前后 / Same subjects, before & after
base = rng.normal(80, 12, 30)                      # 个体差异大 / big between-subject spread
after = base - 2.0 + rng.normal(0, 2.5, 30)        # 真效应 -2kg, 测量噪声小

p_unpaired = st.ttest_ind(base, after, equal_var=False).pvalue
p_paired   = st.ttest_rel(base, after).pvalue

print(f"独立 t (错误用法): p = {p_unpaired:.4f}   ← 个体差异淹没效应")
print(f"配对 t (正确):     p = {p_paired:.2e}   ← 强显著")
print(f"\n相关 r = {np.corrcoef(base, after)[0,1]:.3f} → 配对把 SE 砍掉 ~{(1-np.sqrt(2*(1-np.corrcoef(base,after)[0,1])/2))*100:.0f}%")


<a id="6"></a>
## 6. 比例检验 / Proportion Tests — A/B 的原生形态

转化率比较 = 两比例 z 检验：
$$z = \frac{\hat{p}_A - \hat{p}_B}{\sqrt{\hat{p}(1-\hat{p})\big(\tfrac{1}{n_A} + \tfrac{1}{n_B}\big)}}, \quad \hat{p} = \text{合并比例}$$


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# A/B: 旧按钮 4.8% vs 新按钮 5.6% 转化 / Old 4.8% vs new 5.6% conversion
conv = np.array([480, 560]); n = np.array([10_000, 10_000])
z, p = proportions_ztest(conv, n)
print(f"转化率: A = {conv[0]/n[0]:.2%},  B = {conv[1]/n[1]:.2%}")
print(f"z = {z:.3f},  p = {p:.4f}  → {'显著' if p < 0.05 else '不显著'}")

# 等价: 2×2 卡方 / The equivalent 2x2 chi-square
table = np.array([[480, 9520], [560, 9440]])
chi2, p_chi, *_ = st.chi2_contingency(table)
print(f"卡方 p = {p_chi:.4f}   (z² = {z**2:.2f} = χ² = {chi2:.2f}, 同一个检验)")


<a id="7"></a>
## 7. 卡方检验 / Chi-square Tests

$$\chi^2 = \sum_{\text{cells}} \frac{(O - E)^2}{E}$$

两种用途：
- **拟合优度**：观测频数 vs 理论分布（骰子公平吗？）
- **独立性**：两个类别变量有关联吗（性别 × 是否流失）？期望频数 $E_{ij} = \frac{\text{行和}\times\text{列和}}{n}$

⚠ 适用条件：期望频数全部 ≥5（不满足 → **Fisher 精确检验**）。


In [ ]:
# 独立性: 套餐类型 × 是否流失 / Plan type vs churn
table = pd.DataFrame({"churn": [120, 90, 30], "stay": [880, 1110, 770]},
                     index=["basic", "pro", "enterprise"])
chi2, p, dof, expected = st.chi2_contingency(table)
print(table.assign(churn_rate=(table.churn/(table.churn+table.stay)).round(3)))
print(f"\nχ² = {chi2:.2f}, dof = {dof}, p = {p:.2e} → 套餐与流失{'相关' if p<0.05 else '无关'}")
print(f"最小期望频数 = {expected.min():.1f}  (≥5 ✓, 卡方近似有效)")


<a id="8"></a>
## 8. ANOVA：三组以上 / Three or More Groups

**为什么不能两两 t 检验**：3 组要 3 次比较，假阳率膨胀到 $1-(0.95)^3 \approx 14\%$（2.7 节正题）。ANOVA 一次回答"**至少有一组不同吗**"。

$$F = \frac{\text{组间方差}}{\text{组内方差}} = \frac{SS_B/(k-1)}{SS_W/(n-k)} \sim F_{k-1,\,n-k}$$

$F \approx 1$ → 组间差异和噪声同级 → 无差异。$F \gg 1$ → 有戏。**显著后**才做事后两两比较（Tukey HSD，自带多重校正）。


In [ ]:
# 三种教学法的考试成绩 / Three teaching methods
g1 = rng.normal(70, 10, 40); g2 = rng.normal(74, 10, 40); g3 = rng.normal(71, 10, 40)
f_stat, p = st.f_oneway(g1, g2, g3)
print(f"ANOVA: F = {f_stat:.3f}, p = {p:.4f}")

# 手写 F 验证 / Verify F by hand
allx = np.concatenate([g1, g2, g3]); k, n_ = 3, len(allx)
ss_b = sum(len(g)*(g.mean()-allx.mean())**2 for g in [g1, g2, g3])
ss_w = sum(((g-g.mean())**2).sum() for g in [g1, g2, g3])
print(f"手写:  F = {(ss_b/(k-1)) / (ss_w/(n_-k)):.3f}   ← 一致")

if p < 0.05:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    labels = ["m1"]*40 + ["m2"]*40 + ["m3"]*40
    print("\n", pairwise_tukeyhsd(allx, labels).summary())


<a id="9"></a>
## 9. 非参检验 / Nonparametric Tests

不假设分布形状——**靠秩（rank）**。代价：正态数据下功效略低（ARE ≈ 0.955）；回报：重尾/偏态/有序数据下**更稳更强**。

| 参数版 | 非参版 | 检验什么 |
|---|---|---|
| 独立 t | **Mann-Whitney U** | 一组的值倾向于大于另一组吗（随机优越性）|
| 配对 t | **Wilcoxon signed-rank** | 差值的对称中心 ≠ 0 吗 |
| — | **KS 两样本** | 两个**整个分布**（形状+位置）相同吗 |

**何时切非参**：明显偏态/重尾 + n 小（CLT 没生效）、有序量表（满意度 1-5）、有截断/异常值。


In [ ]:
# 重尾小样本: t vs Mann-Whitney 的功效对比
# Heavy tails, small n: t vs Mann-Whitney power
n_sim, n_ = 4000, 25
hits_t = hits_mw = 0
for _ in range(n_sim):
    a = rng.standard_t(2.5, n_)               # 重尾 / heavy-tailed
    b = rng.standard_t(2.5, n_) + 0.8          # 真位移 +0.8
    hits_t  += st.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    hits_mw += st.mannwhitneyu(a, b).pvalue < 0.05

print(f"重尾数据 (t df=2.5), 真效应 +0.8, n={n_}/组:")
print(f"  Welch t 功效      = {hits_t/n_sim:.1%}")
print(f"  Mann-Whitney 功效 = {hits_mw/n_sim:.1%}   ← 重尾下秩检验反超")


<a id="10"></a>
## 10. 实战：Tips 数据集检验套餐 / Hands-on on Tips

把 0.5/2.1 节 EDA 看出的模式全部**正式检验**——EDA 提出假设，检验回答假设。
EDA generates hypotheses; tests answer them.


In [ ]:
tips = sns.load_dataset("tips")
tips["tip_rate"] = tips.tip / tips.total_bill

# Q1: 周末账单更高吗? (独立 Welch t)
wk = tips[tips.day.isin(["Sat", "Sun"])]["total_bill"]
wd = tips[tips.day.isin(["Thur", "Fri"])]["total_bill"]
r1 = st.ttest_ind(wk, wd, equal_var=False)

# Q2: 吸烟者小费率方差更大吗? (Levene — 比较方差, 对非正态稳健)
sm = tips[tips.smoker == "Yes"]["tip_rate"]; ns = tips[tips.smoker == "No"]["tip_rate"]
r2 = st.levene(sm, ns)

# Q3: 性别和吸烟独立吗? (卡方)
r3 = st.chi2_contingency(pd.crosstab(tips.sex, tips.smoker))

# Q4: 四天的小费率有差异吗? (ANOVA + 非参 Kruskal 对照)
groups = [g["tip_rate"].values for _, g in tips.groupby("day", observed=True)]
r4a = st.f_oneway(*groups); r4b = st.kruskal(*groups)

# Q5: 小费率分布在午餐/晚餐相同吗? (KS)
r5 = st.ks_2samp(tips[tips.time=="Lunch"]["tip_rate"], tips[tips.time=="Dinner"]["tip_rate"])

print(f"{'question':<38} {'test':<16} {'p':>9}   verdict")
print("-" * 80)
for q, t_, p_ in [
    ("周末账单 > 工作日?",        "Welch t",       r1.pvalue),
    ("吸烟者 tip_rate 方差更大?",  "Levene",        r2.pvalue),
    ("性别 × 吸烟 独立?",          "Chi-square",    r3.pvalue),
    ("四天 tip_rate 有差异?",      "ANOVA",         r4a.pvalue),
    ("  (同上, 非参对照)",         "Kruskal",       r4b.pvalue),
    ("午/晚餐 tip_rate 同分布?",   "KS 2-sample",   r5.pvalue),
]:
    print(f"{q:<40} {t_:<16} {p_:>9.4f}   {'拒绝 H0' if p_ < 0.05 else '不拒绝'}")


**读结果的纪律**：
1. 周末账单显著更高（EDA 印象被证实）
2. 吸烟者方差差异显著（0.5 节小提琴图的"长尾"得到正式确认）
3. ANOVA 和 Kruskal 结论一致 → 结论稳健（两种假设体系都同意）
4. ⚠ 我们跑了 6 个检验——**这正是 2.7 节多重比较问题的现场**：6 次检验全 α=0.05，至少一个假阳的概率已达 26%。
We just ran six tests — exactly the multiple-testing crime scene that 2.7 prosecutes.


<a id="11"></a>
## 11. 小结 + 检验选择决策树 / Summary + Decision Tree

```
要比什么?
├── 一组 vs 基准值
│     ├── 均值: 单样本 t        ├── 比例: binomial test
├── 两组
│     ├── 独立 + 数值: Welch t ⭐ (永远 equal_var=False)
│     │     └── 重尾/偏态/小 n → Mann-Whitney
│     ├── 配对: paired t (能配对就配对!)
│     │     └── 非参版 → Wilcoxon signed-rank
│     ├── 比例: 两比例 z (≡ 2×2 卡方)
│     ├── 方差: Levene (不用 F 检验, 对非正态太脆)
│     └── 整个分布: KS 两样本
├── 三组以上
│     ├── ANOVA → 显著后 Tukey HSD
│     └── 非参版 → Kruskal-Wallis
└── 两个类别变量: 卡方独立性 (期望<5 → Fisher 精确)
```

### 💡 面试速查
1. **p 值标准答案**："$H_0$ 为真时，看到这么极端数据的概率"——**不是** $H_0$ 为真的概率
2. **$H_0$ 真 ⇒ p ~ Uniform(0,1)**：5% 假阳的来源
3. **永远 Welch**：scipy 要手动 `equal_var=False`
4. **配对的威力**：协方差吃掉个体差异（CUPED 雏形）
5. **不拒绝 ≠ 证明无效应**：可能只是 power 不足
6. **统计显著 ≠ 业务显著**：n 巨大时一切都显著

### 下一节
**2.7 多重比较**——刚才一口气跑 6 个检验埋的雷，现在引爆：FWER、Bonferroni、BH-FDR。
